# Variational Quantum Circuit

Raw OPM-MEG → Preprocessing → Epoching → Trial tensors → TT decomposition → TT features → Dimensionality reduction → Angle encoding → VQC → Classification

The purpose of the VQC is to determine whether the compressed representation of the MEG trial contains information that can distinguish the four tasks:
{auditory,somatosensory,motor,rest}.

1. Takes 32 MEG trials.
2. Runs them through the quantum circuit.
3. Gets four outputs.
4. Applies softmax.
5. Compares predictions against the true labels.
6. Calculates cross-entropy loss.
7. Uses Adam to update the VQC parameters.

Conceptually:
MEG features → VQC(θ) → prediction → loss → ∂θ/∂L → update θ.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor
from pennylane import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Experiment 1: Initial VQC

EXPERIMENTS:

Layers: 
- Trainable rotations: Rot has three trainable angles.
- Entanglement: CNOT

| Qubits | Layers | Parameters |
| -----: | -----: | ---------: |
|      4 |      2 |         24 |
|      4 |      4 |         48 |
|      4 |      6 |         72 |
|      8 |      2 |         48 |
|      8 |      4 |         96 |
|      8 |      6 |        144 |
|     16 |      2 |         96 |
|     16 |      4 |        192 |
|     16 |      6 |        288 |

- EPOCHS
- BATCH SIZE
- LEARNING RATE


EXPERIMENT 1 SETTINGS:

- 4, 8, 12, 16 qubits/PCA
- N_LAYERS = 2
- N_EPOCHS = 30
- LEARNING_RATE = 0.05
- BATCH_SIZE = 32
- RANDOM_SEED = 42

In [ ]:
# ============================================================
# PATHS
# ============================================================

ANGLE_ROOT = Path(
    "../data/vqc/angle_encoding"
)

RESULTS_ROOT = Path(
    "../results/vqc"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXCEL_PATH = (
    RESULTS_ROOT /
    "vqc_pca_qubit_comparison.xlsx"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

QUBIT_LIST = [
    4,
    8,
    12,
    16
]

N_LAYERS = 2
N_CLASSES = 4
N_EPOCHS = 30
LEARNING_RATE = 0.05
BATCH_SIZE = 32
RANDOM_SEED = 42

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

# RANDOM SEED
np.random.seed(
    RANDOM_SEED
)

# ============================================================
# FUNCTIONS
# ============================================================

def softmax(x):

    x = np.asarray(x)

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        np.sum(
            exp_x,
            axis=-1,
            keepdims=True
        )
    )

def initialise_weights(n_qubits):

    return (
        0.01
        * np.random.randn(
            N_LAYERS,
            n_qubits,
            3
        )
    )

def create_quantum_circuit(n_qubits):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev)
    def quantum_circuit(
        x,
        weights
    ):

        # ANGLE ENCODING
        qml.AngleEmbedding(
            x,
            wires=range(n_qubits),
            rotation="Y"
        )

        for layer in range(N_LAYERS):

            # TRAINABLE ROTATIONS
            for qubit in range(n_qubits):

                # Each Rot gate has three trainable parameters.
                # So 2 layers × 8 qubits × 3 = 48 trainable parameters.
                qml.Rot(
                    weights[layer, qubit, 0],
                    weights[layer, qubit, 1],
                    weights[layer, qubit, 2],
                    wires=qubit
                )

            # ENTANGLEMENT
            for qubit in range(
                n_qubits - 1
            ):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1
                    ]
                )

        # MEASUREMENTS
        return [
            qml.expval(
                qml.PauliZ(i)
            )
            for i in range(N_CLASSES)
        ]

    return quantum_circuit

# FORWARD PASS

def predict_logits(
    X,
    weights,
    quantum_circuit
):

    outputs = []

    for sample in X:

        result = quantum_circuit(
            sample,
            weights
        )

        outputs.append(
            np.asarray(result)
        )

    return np.asarray(
        outputs
    )


# CROSS-ENTROPY

def cross_entropy(
    probabilities,
    labels
):

    probabilities = np.clip(
        probabilities,
        1e-10,
        1.0
    )

    losses = -np.log(
        probabilities[
            np.arange(
                len(labels)
            ),
            labels
        ]
    )

    return np.mean(
        losses
    )

# ============================================================
# TRAIN VQC
# ============================================================

def train_vqc(
    X_train,
    y_train,
    n_qubits
):

    quantum_circuit = (
        create_quantum_circuit(
            n_qubits
        )
    )

    weights = initialise_weights(n_qubits)

    # OPTIMISER
    opt = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )

    # COST FUNCTION
    def cost_fn(
        weights,
        X_batch,
        y_batch
    ):

        logits = predict_logits(
            X_batch,
            weights,
            quantum_circuit
        )

        probabilities = softmax(
            logits
        )

        return cross_entropy(
            probabilities,
            y_batch
        )

    # TRAINING LOOP

    loss_history = []

    for epoch in range(
        N_EPOCHS
    ):

        # Random mini-batch
        batch_size = min(
            BATCH_SIZE,
            len(X_train)
        )

        batch_indices = np.random.choice(
            len(X_train),
            size=batch_size,
            replace=False
        )

        X_batch = X_train[
            batch_indices
        ]

        y_batch = y_train[
            batch_indices
        ]

        # Update weights
        weights, loss = opt.step_and_cost(
            lambda w:
                cost_fn(
                    w,
                    X_batch,
                    y_batch
                ),
            weights
        )

        loss_history.append(
            float(loss)
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:3d}/{N_EPOCHS} "
                f"| Loss = "
                f"{loss:.6f}"
            )

    return (
        weights,
        quantum_circuit,
        loss_history
    )

# ============================================================
# PREDICTION
# ============================================================

def predict(
    X,
    weights,
    quantum_circuit
):

    logits = predict_logits(
        X,
        weights,
        quantum_circuit
    )

    probabilities = softmax(
        logits
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return (
        predictions,
        probabilities
    )

all_results = []

for n_qubits in QUBIT_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC EXPERIMENT: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)

    # Check angle encoding directory

    for test_subject in SUBJECTS:

        print("\n")
        print("-" * 80)

        print(
            f"QUBITS = {n_qubits}"
        )

        print(
            f"TEST SUBJECT = "
            f"{test_subject}"
        )

        print("-" * 80)

        # Load corresponding angle data

        angle_path = (
            ANGLE_ROOT
            /
            f"loso_test_{test_subject}"
            /
            f"angle_{n_qubits}"
            /
            "data.npz"
        )

        if not angle_path.exists():

            raise FileNotFoundError(
                f"\nMissing angle file:\n"
                f"{angle_path}\n\n"
                f"Make sure PCA/angle encoding "
                f"has been generated for "
                f"{n_qubits} components."
            )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        X_train = data[
            "angles_train"
        ]

        X_test = data[
            "angles_test"
        ]

        y_train = data[
            "y_train"
        ]

        y_test = data[
            "y_test"
        ]

        # Check dimensions

        if X_train.shape[1] != n_qubits:

            raise ValueError(
                f"Expected {n_qubits} "
                f"features but received "
                f"{X_train.shape[1]}"
            )

        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )

        print(
            "Training labels:",
            np.unique(y_train)
        )

        print(
            "Testing labels:",
            np.unique(y_test)
        )

        # TRAIN

        print("\n")
        print(
            "Training VQC..."
        )

        (
            weights,
            quantum_circuit,
            loss_history
        ) = train_vqc(
            X_train,
            y_train,
            n_qubits
        )

        # TEST

        print("\n")
        print(
            "Testing VQC..."
        )

        predictions, probabilities = (
            predict(
                X_test,
                weights,
                quantum_circuit
            )
        )

        # METRICS

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )

        macro_f1 = f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=[
                0,
                1,
                2,
                3
            ]
        )

        print(
            "\nTest accuracy:",
            f"{accuracy:.4f}"
        )

        print(
            "Balanced accuracy:",
            f"{balanced_accuracy:.4f}"
        )

        print(
            "Macro F1:",
            f"{macro_f1:.4f}"
        )

        print(
            "Weighted F1:",
            f"{weighted_f1:.4f}"
        )

        print(
            "\nConfusion matrix:"
        )

        print(cm)

        # SAVE INDIVIDUAL RESULT

        result_dir = (
            RESULTS_ROOT
            /
            f"qubits_{n_qubits}"
        )

        result_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            result_dir
            /
            f"loso_test_{test_subject}.npz"
        )

        np.savez_compressed(

            save_path,
            test_subject=test_subject,
            predictions=predictions,
            probabilities=probabilities,
            y_test=y_test,
            accuracy=accuracy,
            balanced_accuracy=balanced_accuracy,
            macro_f1=macro_f1,
            weighted_f1=weighted_f1,
            confusion_matrix=cm,
            weights=weights,
            loss_history=np.asarray(
                loss_history
            ),
            n_qubits=n_qubits,
            n_layers=N_LAYERS
        )

        print(
            "Saved:",
            save_path
        )

        # STORE ROW FOR EXCEL

        all_results.append({

            "subject": test_subject,
            "n_qubits": n_qubits,
            "accuracy": accuracy,
            "balanced_accuracy":
                balanced_accuracy,
            "macro_f1":
                macro_f1,
            "weighted_f1":
                weighted_f1,
            "final_training_loss":
                loss_history[-1]
        })

# CONVERT RESULTS TO DATAFRAME

results_df = pd.DataFrame(
    all_results
)
    
# LOSO SUMMARY

summary_df = (
    results_df
    .groupby(
        "n_qubits"
    )
    .agg({

        "accuracy":
            ["mean", "std"],

        "balanced_accuracy":
            ["mean", "std"],

        "macro_f1":
            ["mean", "std"],

        "weighted_f1":
            ["mean", "std"],

        "final_training_loss":
            ["mean", "std"]

    })
    .reset_index()
)

# Flatten column names

summary_df.columns = [

    "n_qubits",

    "accuracy_mean",
    "accuracy_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "macro_f1_mean",
    "macro_f1_std",

    "weighted_f1_mean",
    "weighted_f1_std",

    "loss_mean",
    "loss_std"

]

# SAVE EXCEL

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="LOSO Results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


print("\n")
print("=" * 80)
print("EXCEL RESULTS SAVED")
print("=" * 80)

print(
    EXCEL_PATH
)

# PRINT SUMMARY

print("\n")
print("=" * 80)
print("VQC QUANTUM DIMENSION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)

# ============================================================
# PLOT 1:
# MEAN ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["accuracy_mean"],
    yerr=summary_df["accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "VQC Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


accuracy_plot = (
    RESULTS_ROOT /
    "accuracy_vs_qubits.png"
)

plt.savefig(
    accuracy_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 2:
# MACRO F1 VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["macro_f1_mean"],
    yerr=summary_df["macro_f1_std"],
    marker="o",
    capsize=5
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "VQC Macro F1 vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()


f1_plot = (
    RESULTS_ROOT /
    "macro_f1_vs_qubits.png"
)

plt.savefig(
    f1_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 3:
# BALANCED ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["balanced_accuracy_mean"],
    yerr=summary_df["balanced_accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Balanced accuracy"
)

plt.title(
    "VQC Balanced Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


balanced_plot = (
    RESULTS_ROOT /
    "balanced_accuracy_vs_qubits.png"
)

plt.savefig(
    balanced_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 4:
# ACCURACY FOR EACH SUBJECT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = results_df[
        results_df["subject"] == subject
    ]

    plt.plot(
        subject_data["n_qubits"],
        subject_data["accuracy"],
        marker="o",
        label=f"Subject {subject}"
    )


plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "LOSO Accuracy Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


subject_plot = (
    RESULTS_ROOT /
    "accuracy_by_subject.png"
)

plt.savefig(
    subject_plot,
    dpi=300
)

plt.close()


# ============================================================
# COMPLETE
# ============================================================

print("\n")
print("=" * 80)
print("ALL VQC EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Plots saved in:",
    RESULTS_ROOT
)



VQC EXPERIMENT: 4 QUBITS


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 002
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.394091
Epoch   5/30 | Loss = 1.320159
Epoch  10/30 | Loss = 1.490590
Epoch  15/30 | Loss = 1.482375
Epoch  20/30 | Loss = 1.367804
Epoch  25/30 | Loss = 1.409850
Epoch  30/30 | Loss = 1.379817


Testing VQC...

Test accuracy: 0.2682
Balanced accuracy: 0.2456
Macro F1: 0.1283
Weighted F1: 0.1323

Confusion matrix:
[[  5 395   0   0]
 [ 11 392   0   2]
 [  2 211   0   4]
 [  1 460   0   1]]
Saved: ../results/vqc/qubits_4/loso_test_002.npz


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 005
--------------------------------------------------------------------------------
Training: 

## Test VQC

VQC TRAINABILITY TEST

In [ ]:
N_QUBITS = 8
N_LAYERS = 2

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    dev,
    interface="autograd"
)
def circuit(x, weights):

    # -----------------------------
    # Angle encoding
    # -----------------------------

    for q in range(N_QUBITS):

        qml.RY(
            x[q],
            wires=q
        )

    # -----------------------------
    # Trainable layers
    # -----------------------------

    for layer in range(N_LAYERS):

        for q in range(N_QUBITS):

            qml.RY(
                weights[layer, q, 0],
                wires=q
            )

            qml.RZ(
                weights[layer, q, 1],
                wires=q
            )

        # Entanglement
        for q in range(N_QUBITS - 1):

            qml.CNOT(
                wires=[
                    q,
                    q + 1
                ]
            )

    return [
        qml.expval(
            qml.PauliZ(q)
        )
        for q in range(N_QUBITS)
    ]


# ============================================================
# CREATE TRAINABLE PARAMETERS
# ============================================================

weights = np.array(
    0.01 * np.random.randn(
        N_LAYERS,
        N_QUBITS,
        2
    ),
    requires_grad=True
)

x = np.array(
    np.random.uniform(
        0,
        np.pi,
        N_QUBITS
    )
)

print("=" * 70)
print("VQC TRAINABILITY TEST")
print("=" * 70)

print(
    "Weights shape:",
    weights.shape
)

print(
    "Weights requires grad:",
    weights.requires_grad
)


# ============================================================
# CIRCUIT OUTPUT
# ============================================================

output = circuit(
    x,
    weights
)

print(
    "\nCircuit output:"
)

print(output)


# ============================================================
# SIMPLE LOSS
# ============================================================

def loss_fn(weights):

    output = circuit(
        x,
        weights
    )

    return qml.math.sum(
        qml.math.stack(output)
    )


loss_before = loss_fn(weights)

print(
    "\nLoss before:",
    loss_before
)


# ============================================================
# GRADIENT
# ============================================================

gradient = qml.grad(
    loss_fn
)(weights)

print(
    "\nGradient shape:",
    gradient.shape
)

print(
    "Gradient norm:",
    np.linalg.norm(gradient)
)


# ============================================================
# TEST OPTIMIZER
# ============================================================

opt = qml.AdamOptimizer(
    stepsize=0.01
)

weights_new, loss_new = opt.step_and_cost(
    loss_fn,
    weights
)

print(
    "\nLoss after one optimizer step:",
    loss_new
)

print(
    "Weight change:",
    np.linalg.norm(
        weights_new - weights
    )
)


if np.linalg.norm(gradient) > 1e-10:

    print(
        "\nPASS: VQC has non-zero gradients."
    )

else:

    print(
        "\nFAIL: VQC gradient is zero."
    )


if np.linalg.norm(
    weights_new - weights
) > 1e-10:

    print(
        "PASS: optimizer changed "
        "the VQC parameters."
    )

else:

    print(
        "FAIL: optimizer did not "
        "change the parameters."
    )

VQC TRAINABILITY TEST
Weights shape: (2, 8, 2)
Weights requires grad: True

Circuit output:
[tensor(-0.99711141, requires_grad=True), tensor(0.03261246, requires_grad=True), tensor(0.95321531, requires_grad=True), tensor(0.01022093, requires_grad=True), tensor(0.43151944, requires_grad=True), tensor(-0.00596899, requires_grad=True), tensor(-0.22071696, requires_grad=True), tensor(0.00589842, requires_grad=True)]

Loss before: 0.20966919547780882

Gradient shape: (2, 8, 2)
Gradient norm: 1.6291960122612252

Loss after one optimizer step: 0.20966919547780882
Weight change: 0.04859651462308379

PASS: VQC has non-zero gradients.
PASS: optimizer changed the VQC parameters.


Test 1 - Class Balance

In [23]:
# ============================================================
# TEST 1: CLASS DISTRIBUTION
# ============================================================

print("="*80)
print("CLASS DISTRIBUTION TEST")
print("="*80)

for subject in SUBJECTS:

    path = (
        ANGLE_ROOT
        /
        f"loso_test_{subject}"
        /
        "angle_8"
        /
        "data.npz"
    )

    data = np.load(path)

    y_train = data["y_train"]
    y_test = data["y_test"]

    print("\nSubject:", subject)

    print("Training:")
    unique, counts = np.unique(
        y_train,
        return_counts=True
    )

    for u,c in zip(unique,counts):
        print(
            f" Class {u}: {c} "
            f"({100*c/len(y_train):.2f}%)"
        )


    print("Testing:")

    unique, counts = np.unique(
        y_test,
        return_counts=True
    )

    for u,c in zip(unique,counts):
        print(
            f" Class {u}: {c} "
            f"({100*c/len(y_test):.2f}%)"
        )

CLASS DISTRIBUTION TEST

Subject: 002
Training:
 Class 0: 1200 (27.47%)
 Class 1: 1224 (28.02%)
 Class 2: 633 (14.49%)
 Class 3: 1312 (30.03%)
Testing:
 Class 0: 400 (26.95%)
 Class 1: 405 (27.29%)
 Class 2: 217 (14.62%)
 Class 3: 462 (31.13%)

Subject: 005
Training:
 Class 0: 1200 (27.47%)
 Class 1: 1229 (28.13%)
 Class 2: 604 (13.82%)
 Class 3: 1336 (30.58%)
Testing:
 Class 0: 400 (26.95%)
 Class 1: 400 (26.95%)
 Class 2: 246 (16.58%)
 Class 3: 438 (29.51%)

Subject: 006
Training:
 Class 0: 1200 (27.01%)
 Class 1: 1222 (27.51%)
 Class 2: 684 (15.40%)
 Class 3: 1336 (30.08%)
Testing:
 Class 0: 400 (28.35%)
 Class 1: 407 (28.84%)
 Class 2: 166 (11.76%)
 Class 3: 438 (31.04%)

Subject: 093
Training:
 Class 0: 1200 (27.40%)
 Class 1: 1212 (27.68%)
 Class 2: 629 (14.36%)
 Class 3: 1338 (30.55%)
Testing:
 Class 0: 400 (27.14%)
 Class 1: 417 (28.29%)
 Class 2: 221 (14.99%)
 Class 3: 436 (29.58%)


Test 2 - VQC Outputs

In [24]:
# ============================================================
# TEST 2: PREDICTION COLLAPSE
# ============================================================

print("="*80)
print("PREDICTION COLLAPSE TEST")
print("="*80)


for q in QUBIT_LIST:

    for subject in SUBJECTS:


        path = (
            RESULTS_ROOT
            /
            f"qubits_{q}"
            /
            f"loso_test_{subject}.npz"
        )


        data = np.load(path)

        predictions = data["predictions"]


        unique, counts = np.unique(
            predictions,
            return_counts=True
        )


        print(
            f"\nQubits {q} | Subject {subject}"
        )

        for u,c in zip(unique,counts):

            print(
                f"Prediction {u}: "
                f"{c} "
                f"({100*c/len(predictions):.2f}%)"
            )

PREDICTION COLLAPSE TEST

Qubits 4 | Subject 002
Prediction 0: 19 (1.28%)
Prediction 1: 1458 (98.25%)
Prediction 3: 7 (0.47%)

Qubits 4 | Subject 005
Prediction 0: 3 (0.20%)
Prediction 1: 1356 (91.37%)
Prediction 2: 1 (0.07%)
Prediction 3: 124 (8.36%)

Qubits 4 | Subject 006
Prediction 0: 3 (0.21%)
Prediction 1: 77 (5.46%)
Prediction 2: 2 (0.14%)
Prediction 3: 1329 (94.19%)

Qubits 4 | Subject 093
Prediction 0: 602 (40.84%)
Prediction 1: 667 (45.25%)
Prediction 2: 6 (0.41%)
Prediction 3: 199 (13.50%)

Qubits 8 | Subject 002
Prediction 0: 1480 (99.73%)
Prediction 1: 2 (0.13%)
Prediction 3: 2 (0.13%)

Qubits 8 | Subject 005
Prediction 0: 96 (6.47%)
Prediction 1: 117 (7.88%)
Prediction 2: 1 (0.07%)
Prediction 3: 1270 (85.58%)

Qubits 8 | Subject 006
Prediction 0: 226 (16.02%)
Prediction 1: 203 (14.39%)
Prediction 2: 2 (0.14%)
Prediction 3: 980 (69.45%)

Qubits 8 | Subject 093
Prediction 0: 652 (44.23%)
Prediction 1: 387 (26.26%)
Prediction 2: 24 (1.63%)
Prediction 3: 411 (27.88%)

Qubits 

Test 3

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


print("="*80)
print("CLASSICAL BASELINE")
print("="*80)


for subject in SUBJECTS:


    path = (
        ANGLE_ROOT
        /
        f"loso_test_{subject}"
        /
        "angle_8"
        /
        "data.npz"
    )


    data = np.load(path)


    X_train=data["angles_train"]
    X_test=data["angles_test"]

    y_train=data["y_train"]
    y_test=data["y_test"]


    clf = LogisticRegression(
        max_iter=1000
    )


    clf.fit(
        X_train,
        y_train
    )


    pred = clf.predict(
        X_test
    )


    acc = accuracy_score(
        y_test,
        pred
    )


    f1 = f1_score(
        y_test,
        pred,
        average="macro"
    )


    print(
        subject,
        "Accuracy:",
        acc,
        "Macro F1:",
        f1
    )

CLASSICAL BASELINE
002 Accuracy: 0.316711590296496 Macro F1: 0.18241756382154023
005 Accuracy: 0.3106469002695418 Macro F1: 0.21021917053980643
006 Accuracy: 0.293408929836995 Macro F1: 0.2064100414770636
093 Accuracy: 0.3331071913161465 Macro F1: 0.24528845037478747


Test 4 - Overfit

In [ ]:
# ============================================================
# TEST 4: OVERFIT TEST
# ============================================================


subject="002"
q=8


path=(
    ANGLE_ROOT
    /
    f"loso_test_{subject}"
    /
    f"angle_{q}"
    /
    "data.npz"
)


data=np.load(path)


X=data["angles_train"]
y=data["y_train"]


weights,circuit,loss=train_vqc(
    X,
    y,
    q
)


pred,_=predict(
    X,
    weights,
    circuit
)


train_acc=accuracy_score(
    y,
    pred
)


print(
    "Training accuracy:",
    train_acc
)

Epoch   1/30 | Loss = 1.430320
Epoch   5/30 | Loss = 1.432528
Epoch  10/30 | Loss = 1.319325
Epoch  15/30 | Loss = 1.360122
Epoch  20/30 | Loss = 1.347962
Epoch  25/30 | Loss = 1.380705
Epoch  30/30 | Loss = 1.335456
Training accuracy: 0.2993820096131838


In [ ]:
# ============================================================
# TEST 5: WEIGHT CHANGE
# ============================================================


initial = initialise_weights(8)


trained, circuit, loss = train_vqc(
    X_train,
    y_train,
    8
)


difference = np.linalg.norm(
    trained-initial
)


print(
    "Weight change:",
    difference
)

Epoch   1/30 | Loss = 1.384204
Epoch   5/30 | Loss = 1.474832
Epoch  10/30 | Loss = 1.343338
Epoch  15/30 | Loss = 1.387859
Epoch  20/30 | Loss = 1.416469
Epoch  25/30 | Loss = 1.390327
Epoch  30/30 | Loss = 1.346557
Weight change: 2.194158132404436


In [9]:
for n_qubits in [4,8,12,16]:

    print("\n")
    print("="*50)
    print("Testing", n_qubits, "qubits")

    circuit = make_vqc(
        n_qubits,
        N_LAYERS
    )

    weights = qml.numpy.array(
        np.random.normal(
            0,
            0.05,
            size=(
                N_LAYERS,
                n_qubits,
                2
            )
        ),
        requires_grad=True
    )


    angles = np.random.uniform(
        0,
        np.pi,
        size=n_qubits
    )


    print(
        "Weights:",
        weights.shape
    )

    print(
        "Angles:",
        angles.shape
    )


    output = circuit(
        angles,
        weights
    )


    print(
        "Output:",
        len(output)
    )



Testing 4 qubits
Weights: (2, 4, 2)
Angles: (4,)
Output: 4


Testing 8 qubits
Weights: (2, 8, 2)
Angles: (8,)
Output: 8


Testing 12 qubits
Weights: (2, 12, 2)
Angles: (12,)
Output: 12


Testing 16 qubits
Weights: (2, 16, 2)
Angles: (16,)
Output: 16
